# CIGALE Decomposition Validation: A Population-Weighted AGN Template for the Theoretical Grid

`CIGALE_Decomposition_Validation_GeometryFix.ipynb` showed that
reconstructing each galaxy with CIGALE's *own* best-fit SKIRTOR geometry
(matched per galaxy via `agn.i`) fixes the Quiescent-region classification
asymmetry seen with the paper's fixed Type 1/Type 2 templates. But that
"Matched" approach needs each galaxy's own CIGALE fit result - it cannot be
used in the paper's Part 1 *theoretical* composite-SED grid
(`composite_math.create_composite_sed` swept over `config.ALPHA_VALUES`),
which by design has no real galaxy to match against; it generates purely
synthetic composites.

**This notebook asks a follow-up question:** is there a single,
theory-usable AGN template - not tied to any specific real galaxy - that
captures most of the Matched approach's improvement? The natural candidate:
a **population-weighted blend** of the two SKIRTOR geometries CIGALE
actually fit (`i=30`, 76% of AGN-host galaxies; `i=70`, 24%), combined into
one template via `flux_blend = w30 * flux_i30 + w70 * flux_i70`. Both
templates share an identical wavelength grid (verified below), so this
blend is a simple, well-defined weighted average - not an approximation
requiring interpolation.

This blended template *is* usable in the theoretical grid exactly like
Type1/Type2 (same `create_composite_sed` call, no per-galaxy information
needed), so if it performs close to "Matched," it's a genuine candidate
third reference case for the paper's Part 1 modelling - not just a
diagnostic tool.

This notebook is additive: it does not modify
`CIGALE_Decomposition_Validation.ipynb` or
`CIGALE_Decomposition_Validation_GeometryFix.ipynb`. Where useful, it loads
those notebooks' cached results for direct comparison rather than
recomputing them.

In [ ]:
import sys
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

from src import config
from glass import data_io, composite_math, photometry, visualization, analysis

plt.style.use('default')
visualization.apply_pasa_style()
os.makedirs(config.PROCESSED_DATA_DIR, exist_ok=True)

BLEND_OUTPUT_DIR = os.path.join(config.PROCESSED_DATA_DIR, 'cigale_weighted_blend')
os.makedirs(BLEND_OUTPUT_DIR, exist_ok=True)

GEOMETRY_FIX_DIR = os.path.join(config.PROCESSED_DATA_DIR, 'cigale_geometry_fix')

METHOD_COLORS = {'Type1': '#1A6FB5', 'Type2': '#CC2929', 'Blend': '#8E44AD', 'Matched (ref.)': '#2E8B57'}

## 1. Population and the blended template

Reuses the population and the per-galaxy SKIRTOR-geometry cache built in
`CIGALE_Decomposition_Validation_GeometryFix.ipynb` (`agn_geometry.csv`,
listing each galaxy's CIGALE best-fit `agn.i`) to compute the population
weights `w30`/`w70`.

In [ ]:
cigale_csv = os.path.join(config.RAW_DATA_DIR, 'full_zfourge_decomposed', 'zfourge_full_final.csv')
agn_frac_csv = os.path.join(config.RAW_DATA_DIR, 'full_zfourge_decomposed', 'agn_fractions.csv')
geometry_csv = os.path.join(GEOMETRY_FIX_DIR, 'agn_geometry.csv')

df_cig = pd.read_csv(cigale_csv, low_memory=False)
z_col = 'zpk_x' if 'zpk_x' in df_cig.columns else 'zpk'
df_cig = df_cig.merge(pd.read_csv(agn_frac_csv), on='ID', how='left')

AGN_FRAC_MIN = 0.0
df_agn = df_cig[df_cig['fracAGN'] > AGN_FRAC_MIN].reset_index(drop=True)

if not os.path.exists(geometry_csv):
    raise FileNotFoundError(
        "agn_geometry.csv not found - run CIGALE_Decomposition_Validation_GeometryFix.ipynb "
        "first to build it (a one-time header scan of all AGN-host galaxies)."
    )
geom_df = pd.read_csv(geometry_csv)
df_agn = df_agn.merge(geom_df, on='ID', how='left')

n30 = (df_agn['i'] == 30).sum()
n70 = (df_agn['i'] == 70).sum()
W30 = n30 / (n30 + n70)
W70 = n70 / (n30 + n70)
print(f"AGN-hosting galaxies: {len(df_agn)}; i=30: {n30} ({W30:.1%}), i=70: {n70} ({W70:.1%})")


def _fits_path(gid, field):
    fits_num = gid.split('_', 1)[1]
    return os.path.join(config.RAW_DATA_DIR, 'full_zfourge_decomposed',
                         f"{field.lower()}_best_models_fits", f"{fits_num}_best_model.fits")

In [ ]:
skirtor_dir = os.path.join(config.RAW_DATA_DIR, 'Templates', 'Skirtor')
agn_type1 = data_io.read_skirtor_model(skirtor_dir, **config.SKIRTOR_TYPE1_PARAMS)
agn_type2 = data_io.read_skirtor_model(skirtor_dir, **config.SKIRTOR_TYPE2_PARAMS)

MATCHED_GEOMETRY_PARAMS = {'optical_depth': 7, 'p': 1, 'q': 1, 'opening_angle': 40, 'radius_ratio': 20}
skirtor_i30 = data_io.read_skirtor_model(skirtor_dir, inclination=30, **MATCHED_GEOMETRY_PARAMS)
skirtor_i70 = data_io.read_skirtor_model(skirtor_dir, inclination=70, **MATCHED_GEOMETRY_PARAMS)

assert np.array_equal(skirtor_i30['lambda (Angstroms)'].values, skirtor_i70['lambda (Angstroms)'].values), \
    "i=30 and i=70 templates must share a wavelength grid for a direct weighted blend"

agn_blend = skirtor_i30.copy()
agn_blend['Total Flux (erg/s/cm^2/Angstrom)'] = (
    W30 * skirtor_i30['Total Flux (erg/s/cm^2/Angstrom)'] + W70 * skirtor_i70['Total Flux (erg/s/cm^2/Angstrom)']
)

FIXED_TEMPLATES = {'Type1': agn_type1, 'Type2': agn_type2, 'Blend': agn_blend}
filters = photometry.load_passbands(config.FILTER_PATHS)

print(f"Blend template = {W30:.1%} x (i=30 SKIRTOR) + {W70:.1%} x (i=70 SKIRTOR), "
      f"both t=7,p=1,q=1,oa=40,R=20 - CIGALE's own preferred torus shape, population-weighted "
      f"by how often each inclination was actually fit.")

**Column-reuse note (as in the earlier validation notebooks):**
`data_io.read_cigale_best_model()`'s Fnu-derived `'Total Flux
(erg/s/cm^2/Angstrom)'` column is deliberately overwritten by
`analysis.decompose_cigale_sed(..., target='host')`'s `L_lambda_total`-based
value; the ground-truth "full" SED for every residual below is read from
`L_lambda_total` directly, before decomposition. Relative flux residuals are
only evaluated where true flux exceeds `1e-4` of that galaxy's peak
`L_lambda_total` and restframe wavelength exceeds `1300` A (excludes the
Lyman-continuum/IGM-absorption regime no additive model replicates, and
which plays no role in the U/V/J passbands).

In [ ]:
FLUX_FLOOR_FRACTION = 1e-4
WAVELENGTH_VALID_MIN = 1300.0


def process_galaxy(gid, field, z, fracAGN):
    '''
    Reconstructs galaxy `gid`'s full SED via Type1, Type2 (paper's fixed
    templates) and Blend (the population-weighted CIGALE-consistent
    template built above) - all three usable without any per-galaxy CIGALE
    information, unlike the "Matched" approach.
    '''
    path = _fits_path(gid, field)
    if not os.path.exists(path):
        return None

    full_sed = data_io.read_cigale_best_model(path, redshift=z, restframe=True)
    wl_full = full_sed['lambda (Angstroms)'].values.astype(float)
    full_L = full_sed['L_lambda_total'].values.astype(float)
    peak_L = np.nanmax(full_L)
    floor = FLUX_FLOOR_FRACTION * peak_L if peak_L > 0 else 0.0

    host_sed = analysis.decompose_cigale_sed(full_sed, target='host')
    alpha_theory = fracAGN / (1.0 - fracAGN) if fracAGN < 1.0 else np.nan

    row = {'ID': gid, 'field': field, 'fracAGN': fracAGN, 'redshift': z, 'alpha_theory': alpha_theory}

    for mname, agn_template in FIXED_TEMPLATES.items():
        composite = composite_math.create_composite_sed(agn_template, host_sed, alpha_theory)
        wl_c = composite['lambda (Angstroms)'].values
        flux_c = composite['Total Flux (erg/s/cm^2/Angstrom)'].values

        full_interp = np.interp(wl_c, wl_full, full_L, left=np.nan, right=np.nan)
        valid = np.isfinite(full_interp) & (full_interp > floor) & (wl_c > WAVELENGTH_VALID_MIN)
        resid = (flux_c[valid] - full_interp[valid]) / full_interp[valid] if valid.any() else np.array([])
        row[f'resid_median_{mname}'] = np.median(resid) if len(resid) else np.nan
        row[f'max_abs_resid_{mname}'] = np.max(np.abs(resid)) if len(resid) else np.nan

        try:
            uv_r, vj_r = photometry.calculate_UVJ_colours(composite, filters['U'], filters['V'], filters['J'])
            cls_r = int(photometry.classify_uvj(np.array([vj_r]), np.array([uv_r]))[0])
        except (ValueError, ZeroDivisionError):
            uv_r, vj_r, cls_r = np.nan, np.nan, -1
        row[f'UV_recombined_{mname}'] = uv_r
        row[f'VJ_recombined_{mname}'] = vj_r
        row[f'cls_recombined_{mname}'] = cls_r

    return row

In [ ]:
SUMMARY_CSV = os.path.join(BLEND_OUTPUT_DIR, 'weighted_blend_summary.csv')

if os.path.exists(SUMMARY_CSV):
    summary_df = pd.read_csv(SUMMARY_CSV)
    print(f"Loaded cached results for {len(summary_df)} galaxies.")
else:
    rows = []
    t_start = time.time()
    n_missing = 0
    for _, r in df_agn.iterrows():
        row = process_galaxy(r['ID'], r['field'], r[z_col], r['fracAGN'])
        if row is None:
            n_missing += 1
            continue
        rows.append(row)
        if len(rows) % 500 == 0:
            print(f"  {len(rows)}/{len(df_agn)} galaxies processed ({time.time() - t_start:.0f}s elapsed)")

    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(SUMMARY_CSV, index=False)
    print(f"Done: {len(summary_df)} galaxies processed, {n_missing} missing FITS files skipped, "
          f"{time.time() - t_start:.0f}s total.")

## 2. Headline result: does the blended template close most of the Matched-vs-fixed gap? (Figure 1)

Compares Type1, Type2, and Blend against the real `UV_Full`/`VJ_Full`
classification, broken down by true UVJ region - and overlays the
per-galaxy "Matched" result (loaded from
`CIGALE_Decomposition_Validation_GeometryFix.ipynb`'s cache, not
recomputed) as a reference ceiling: Blend cannot beat Matched, since Matched
uses information (each galaxy's own CIGALE fit) that Blend deliberately
does not use - the question is how close Blend gets.

In [ ]:
def bootstrap_ci(values, stat_fn=np.mean, n_boot=2000, seed=42, ci=(2.5, 97.5)):
    '''Non-parametric percentile bootstrap CI (docs/figure9_10_bootstrap_methodology.md convention).'''
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    point = stat_fn(values)
    rng = np.random.default_rng(seed)
    boot = rng.choice(values, size=(n_boot, len(values)), replace=True)
    boot_stat = stat_fn(boot, axis=1)
    lo, hi = np.percentile(boot_stat, ci)
    return point, lo, hi


catalog_cols = summary_df.merge(df_cig[['ID', 'UV_Full', 'VJ_Full']], on='ID', how='left')
catalog_cols['cls_full'] = photometry.classify_uvj(catalog_cols['VJ_Full'], catalog_cols['UV_Full'])

METHODS = ['Type1', 'Type2', 'Blend']
region_names = {0: 'Quiescent', 1: 'Star-forming', 2: 'Dusty'}

# Reference-only: Matched results from the prior notebook's cache, for comparison.
matched_csv = os.path.join(GEOMETRY_FIX_DIR, 'geometry_fix_summary.csv')
matched_ref = None
if os.path.exists(matched_csv):
    m = pd.read_csv(matched_csv)[['ID', 'cls_recombined_Matched']]
    catalog_cols = catalog_cols.merge(m, on='ID', how='left')
    matched_ref = 'cls_recombined_Matched'
    METHODS_ALL = METHODS + ['Matched (ref.)']
else:
    METHODS_ALL = METHODS
    print("Matched reference not found - run CIGALE_Decomposition_Validation_GeometryFix.ipynb first "
          "for the comparison overlay (Blend results below are unaffected).")

agreement_table = {}
for mname in METHODS_ALL:
    col = matched_ref if mname == 'Matched (ref.)' else f'cls_recombined_{mname}'
    ok = catalog_cols[col] != -1
    agreement_table[mname] = {}
    for cid in [0, 1, 2]:
        mask = ok & (catalog_cols['cls_full'] == cid)
        correct = (catalog_cols.loc[mask, col] == cid).astype(float)
        p, lo, hi = bootstrap_ci(correct, np.mean, seed=100 + cid)
        agreement_table[mname][cid] = (p, lo, hi, mask.sum())

fig, ax = plt.subplots(figsize=visualization.PASA_WIDE)
x = np.arange(3)
n_methods = len(METHODS_ALL)
width = 0.8 / n_methods
for i_m, mname in enumerate(METHODS_ALL):
    pts = [agreement_table[mname][cid][0] for cid in [0, 1, 2]]
    los = [agreement_table[mname][cid][1] for cid in [0, 1, 2]]
    his = [agreement_table[mname][cid][2] for cid in [0, 1, 2]]
    err = [np.array(pts) - np.array(los), np.array(his) - np.array(pts)]
    ax.bar(x + (i_m - (n_methods - 1) / 2) * width, pts, width, yerr=err, capsize=2,
           color=METHOD_COLORS[mname], label=mname)
ax.set_xticks(x)
ax.set_xticklabels([region_names[c] for c in [0, 1, 2]])
ax.set_ylabel('UVJ classification agreement (reconstructed vs Full)')
ax.set_ylim(0, 1.05)
ax.set_title('Population-weighted blend vs fixed Type1/Type2 (Matched shown as reference ceiling)')
ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(os.path.join(BLEND_OUTPUT_DIR, 'Figure1_blend_vs_fixed_agreement.png'), dpi=300, bbox_inches='tight')
plt.show()

print("Per-region classification agreement (point [95% CI], n):")
for mname in METHODS_ALL:
    print(f"\n{mname}:")
    for cid in [0, 1, 2]:
        p, lo, hi, n = agreement_table[mname][cid]
        print(f"  {region_names[cid]:14s} n={n:5d}  {p:.1%} [{lo:.1%}, {hi:.1%}]")

for mname in METHODS_ALL:
    col = matched_ref if mname == 'Matched (ref.)' else f'cls_recombined_{mname}'
    ok = catalog_cols[col] != -1
    fp = ok & (catalog_cols['cls_full'] != 0) & (catalog_cols[col] == 0)
    fn = ok & (catalog_cols['cls_full'] == 0) & (catalog_cols[col] != 0)
    print(f"\n{mname}: false-positive Quiescent rate = {fp.sum() / ok.sum():.2%} ({fp.sum()}/{ok.sum()}), "
          f"false-negative Quiescent rate (of true Quiescent) = "
          f"{fn.sum() / (catalog_cols['cls_full'] == 0).sum():.1%} "
          f"({fn.sum()}/{(catalog_cols['cls_full'] == 0).sum()})")

## 3. Flux- and colour-level fidelity (Figure 2)

Same check as the geometry-fix notebook: does Blend's classification
improvement come at the cost of worse flux/colour reconstruction overall?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=visualization.PASA_WIDE)

ax = axes[0]
for mname in METHODS:
    vals = summary_df[f'max_abs_resid_{mname}'].replace([np.inf, -np.inf], np.nan).dropna()
    vals = vals[vals > 0]
    ax.hist(np.log10(vals), bins=50, color=METHOD_COLORS[mname], alpha=0.5, label=mname, density=True,
            histtype='step', linewidth=1.5)
ax.set_xlabel('log10(per-galaxy max |relative residual|), wavelength > 1300 A')
ax.set_ylabel('Density')
ax.set_title('Tier 1: flux reconstruction fidelity')
ax.legend(fontsize=7)

ax = axes[1]
for mname in METHODS:
    duv = catalog_cols[f'UV_recombined_{mname}'] - catalog_cols['UV_Full']
    dvj = catalog_cols[f'VJ_recombined_{mname}'] - catalog_cols['VJ_Full']
    combined = pd.concat([duv.abs(), dvj.abs()])
    ax.hist(combined.clip(upper=0.5), bins=60, histtype='step', color=METHOD_COLORS[mname],
            label=mname, density=True, linewidth=1.5)
ax.set_xlabel('|dUV| or |dVJ| (mag)')
ax.set_ylabel('Density')
ax.set_title('Tier 2: colour reconstruction fidelity')
ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(os.path.join(BLEND_OUTPUT_DIR, 'Figure2_blend_flux_colour_fidelity.png'), dpi=300, bbox_inches='tight')
plt.show()

for mname in METHODS:
    duv = catalog_cols[f'UV_recombined_{mname}'] - catalog_cols['UV_Full']
    dvj = catalog_cols[f'VJ_recombined_{mname}'] - catalog_cols['VJ_Full']
    m_uv, lo_uv, hi_uv = bootstrap_ci(np.abs(duv), np.mean, seed=201)
    m_vj, lo_vj, hi_vj = bootstrap_ci(np.abs(dvj), np.mean, seed=211)
    med_resid = summary_df[f'resid_median_{mname}'].median()
    print(f"{mname}: mean|dUV|={m_uv:.4f} [{lo_uv:.4f},{hi_uv:.4f}], "
          f"mean|dVJ|={m_vj:.4f} [{lo_vj:.4f},{hi_vj:.4f}], "
          f"population median flux residual={med_resid:+.4f}")

## 4. Verification: single-galaxy spot-check

Overlays Type1, Type2, and Blend reconstructions against the real Full SED
for a true-Quiescent galaxy, to visualise how the blended template's
optical/UV contribution sits between the two fixed extremes.

In [ ]:
candidates = catalog_cols[(catalog_cols['cls_full'] == 0) & (catalog_cols['cls_recombined_Blend'] == 0)]
if len(candidates) == 0:
    print("No suitable spot-check galaxy found.")
else:
    spot = candidates.iloc[0]
    spot_id, field = spot['ID'], spot['field']
    path = _fits_path(spot_id, field)
    full_sed = data_io.read_cigale_best_model(path, redshift=spot['redshift'], restframe=True)
    host_sed = analysis.decompose_cigale_sed(full_sed, target='host')

    fig, ax = plt.subplots(figsize=visualization.PASA_WIDE)
    ax.loglog(full_sed['lambda (Angstroms)'], full_sed['L_lambda_total'], color='k', lw=1.5, label='Full (CIGALE)')
    ax.loglog(host_sed['lambda (Angstroms)'], host_sed['Total Flux (erg/s/cm^2/Angstrom)'], color='gray', ls='--',
               label='Host (decomposed)')
    for mname, tmpl in FIXED_TEMPLATES.items():
        composite = composite_math.create_composite_sed(tmpl, host_sed, spot['alpha_theory'])
        ax.loglog(composite['lambda (Angstroms)'], composite['Total Flux (erg/s/cm^2/Angstrom)'],
                   color=METHOD_COLORS[mname], ls=':', label=f'Host + {mname}')
    ax.axvspan(3000, 13000, color='yellow', alpha=0.08, label='approx. U-J restframe range')
    ax.set_xlabel('Restframe wavelength (A)')
    ax.set_ylabel('Flux (L_lambda_total units)')
    ax.set_title(f"{spot_id}: true Quiescent (fracAGN={spot['fracAGN']:.2f})")
    ax.legend(fontsize=7)
    plt.tight_layout()
    fig.savefig(os.path.join(BLEND_OUTPUT_DIR, 'Figure3_spotcheck_overlay.png'), dpi=300, bbox_inches='tight')
    plt.show()

    print(f"cls_full=Quiescent; cls_recombined: Type1={spot['cls_recombined_Type1']}, "
          f"Type2={spot['cls_recombined_Type2']}, Blend={spot['cls_recombined_Blend']} (0=Quiescent)")

## 5. Conclusion

The population-weighted blend (`W30 x SKIRTOR(i=30) + W70 x SKIRTOR(i=70)`,
weights set by how often CIGALE actually fit each inclination) is a single,
theory-usable AGN template - no per-galaxy information required, so it
plugs directly into `create_composite_sed` calls in the paper's Part 1 grid
exactly like Type1/Type2. Compare Figure 1's Blend bars against the Matched
reference ceiling above to see how much of the per-galaxy fix's improvement
survives the move from "per-galaxy exact match" to "one population-average
template."

**Caveat carried over from the geometry-fix notebook:** the 76%/24% weights
are specific to *this* CIGALE run's restricted geometry grid (only `i=30`
and `i=70` were ever explored) - they are not a universal AGN inclination
distribution. If the paper's CIGALE fitting is ever re-run with a broader
geometry grid, these weights (and the blend itself) would need
recomputing from the new `agn_geometry.csv`.

This notebook does not modify `CIGALE_Decomposition_Analysis.ipynb`,
`CIGALE_Decomposition_Validation.ipynb`,
`CIGALE_Decomposition_Validation_GeometryFix.ipynb`, or any code in the
`glass` package. Adopting the blended template as an actual third reference
case in the paper's Part 1 modelling (e.g. adding it to
`config.SKIRTOR_TYPE1_PARAMS`-style constants and re-running
`recreate_theoretical_results.py` / `Paper_Results_Master.ipynb`) is a
decision for the paper's author, not made here.